In [0]:
%run "../SparkStreaming/01-streaming-word-count-refactor"

In [0]:
from pyspark.sql.functions import *
import shutil

In [0]:
class batchWCTestCase():
    def __init__(self):
        self.base_dir_loc = "/Workspace/Users/dopetechied@gmail.com/SparkStreaming"
        self.base_vol_loc =  "/Volumes/streamingdata/dbo"

    def cleanTests(self):
        # Drop the table
        spark.sql("DROP TABLE IF EXISTS streamingdata.dbo.word_count_table")
        
        # Drop and recreate checkpoint volume
        spark.sql("DROP VOLUME IF EXISTS streamingdata.dbo.checkpoint")
        spark.sql("CREATE VOLUME IF NOT EXISTS streamingdata.dbo.checkpoint")
        
        # Drop and recreate data volume
        spark.sql("DROP VOLUME IF EXISTS streamingdata.dbo.data")
        spark.sql("CREATE VOLUME IF NOT EXISTS streamingdata.dbo.data")

        # files = dbutils.fs.ls(f"{self.base_vol_loc}/data/")
        # print(f"Files in data volume after clean: {files}")
        print("Done\n")

    def ingestData(self, itr):
        print(f"\tStarting Ingestion...", end = "")
        shutil.copy(f"{self.base_vol_loc}/raw/text_data_{itr}.txt", f"{self.base_vol_loc}/data/text_data_{itr}.txt")
        # print(f"\n\tFiles in volume after ingest:")
        # display(dbutils.fs.ls(f"{self.base_vol_loc}/data/"))
        print("Done\n")

    def assertResult(self, expected_count):
        actual_count = spark.sql("select sum(count) from streamingdata.dbo.word_count_table where substr(word, 1, 1) = 's'").collect()[0][0] or 0
        assert expected_count == actual_count, f"Test Failed! actual count is {actual_count}"

    def runTests(self):
        self.cleanTests()
        wc = batchWC()

        print("Testing first iteration of batch word count...")
        self.ingestData(1)
        wc.wordCount()
        self.assertResult(25)
        print("First Iteration of batch word count completed \n")

        print("Testing second iteration of batch word count...")
        self.ingestData(2)
        wc.wordCount()
        self.assertResult(32)
        print("Second Iteration of batch word count completed \n")

        print("Testing third iteration of batch word count...")
        self.ingestData(3)
        wc.wordCount()
        self.assertResult(37)
        print("Third iteration of batch word count completed \n")





In [0]:
# bwcTC = batchWCTestCase()
# bwcTC.runTests()

In [0]:
class streamWCTestCase():
    def __init__(self):
        self.base_dir_loc = "/Workspace/Users/dopetechied@gmail.com/SparkStreaming"
        self.base_vol_loc =  "/Volumes/streamingdata/dbo"

    def cleanTests(self):
        # Drop the table
        spark.sql("DROP TABLE IF EXISTS streamingdata.dbo.word_count_table")
        
        # Drop and recreate checkpoint volume
        spark.sql("DROP VOLUME IF EXISTS streamingdata.dbo.checkpoint")
        spark.sql("CREATE VOLUME IF NOT EXISTS streamingdata.dbo.checkpoint")
        
        # Drop and recreate data volume
        spark.sql("DROP VOLUME IF EXISTS streamingdata.dbo.data")
        spark.sql("CREATE VOLUME IF NOT EXISTS streamingdata.dbo.data")

        # files = dbutils.fs.ls(f"{self.base_vol_loc}/data/")
        # print(f"Files in data volume after clean: {files}")
        print("Done\n")

    def ingestData(self, itr):
        print(f"\tStarting Ingestion...", end = "")
        shutil.copy(f"{self.base_vol_loc}/raw/text_data_{itr}.txt", f"{self.base_vol_loc}/data/text_data_{itr}.txt")
        # print(f"\n\tFiles in volume after ingest:")
        # display(dbutils.fs.ls(f"{self.base_vol_loc}/data/"))
        print("Done\n")

    def assertResult(self, expected_count):
        actual_count = spark.sql("select sum(count) from streamingdata.dbo.word_count_table where substr(word, 1, 1) = 's'").collect()[0][0] or 0
        assert expected_count == actual_count, f"Test Failed! actual count is {actual_count}"

    def runTests(self):
        import time
        sleepTime = 30

        self.cleanTests()
        wc = streamWC()

        print("Testing first iteration of batch word count...")
        self.ingestData(1)
        sQuery = wc.wordCount()          # Start query — runs and stops
        sQuery.awaitTermination()        # Wait for it to finish
        self.assertResult(25)
        print("First Iteration of batch word count completed \n")

        print("Testing second iteration of batch word count...")
        self.ingestData(2)
        sQuery = wc.wordCount()          # Re-trigger for new data
        sQuery.awaitTermination()
        self.assertResult(32)
        print("Second Iteration of batch word count completed \n")

        print("Testing third iteration of batch word count...")
        self.ingestData(3)
        sQuery = wc.wordCount()
        sQuery.awaitTermination()
        self.assertResult(37)
        print("Third iteration of batch word count completed \n")

        sQuery.stop()



In [0]:
swcTs = streamWCTestCase()
swcTs.runTests()